# Tahap 2: Exploratory Data Analysis (EDA)
**Mata Kuliah:** Data Science 2026  
**Topik:** King County Real Estate Sales & Property Assessment (Harvard Dataverse)  
**Tujuan:** Memahami struktur data transaksi, karakteristik sebaran harga properti, korelasi fitur fisik dan spasial terhadap harga jual, serta memetakan strategi pembersihan data kotor.

## 1. Import Pustaka & Konfigurasi Lingkungan

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Konfigurasi visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

## 2. Memuat Dataset Mentah (*Raw Data*)

In [ ]:
# Memuat dataset resmi transaksi properti King County
DATA_PATH = "../data/raw/02_king_county_real_estate/cleansales.tab"

df = pd.read_csv(DATA_PATH, sep='\t', low_memory=False)
print(f"Dataset berhasil dimuat: {df.shape[0]:,} baris x {df.shape[1]} kolom.")
df.head()

## 3. Inspeksi Awal Karakteristik Data
Periksa dimensi data, tipe data per kelompok kolom, dan cuplikan informasi umum.

In [ ]:
print("--- Informasi Ringkas Struktur Data ---")
print(f"Jumlah Baris Transaksi : {df.shape[0]:,}")
print(f"Jumlah Kolom Atribut   : {df.shape[1]}")

# Distribusi tipe data
print("\nDistribusi Tipe Data:")
print(df.dtypes.value_counts())

## 4. Evaluasi Kualitas Data, Duplikasi & Missing Values
Identifikasi jumlah dan sebaran nilai kosong pada seluruh 147 kolom.

In [ ]:
# Evaluasi nilai kosong
missing_series = df.isnull().sum()
cols_with_missing = missing_series[missing_series > 0]

print(f"Jumlah kolom yang memiliki missing values: {len(cols_with_missing)}")
missing_df = pd.DataFrame({
    'Missing Values': cols_with_missing,
    'Persentase (%)': (cols_with_missing / len(df) * 100).round(2)
}).sort_values(by='Missing Values', ascending=False)

display(missing_df.head(15))

In [ ]:
# Evaluasi spesifik pada kolom ZipCode dan teks
print("Missing ZipCode:", df['ZipCode'].isnull().sum(), f"({df['ZipCode'].isnull().mean()*100:.2f}%)")
print("Cek kelengkapan koordinat spasial:")
print("Missing X (Easting) :", df['X'].isnull().sum())
print("Missing Y (Northing):", df['Y'].isnull().sum())

## 5. Analisis Statistik Deskriptif & Distribusi Target (`SalePrice`)
Eksplorasi ringkasan statistik dan bentuk distribusi harga properti (apakah terdistribusi normal atau menceng/skewed).

In [ ]:
print("--- Statistik Deskriptif Target: SalePrice ---")
display(df['SalePrice'].describe())

# Visualisasi distribusi harga normal vs transformasi logaritmik
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Skala Dolar Asli
sns.histplot(df['SalePrice'] / 1000, kde=True, ax=axes[0], color='#1f77b4', bins=60)
axes[0].set_title("Distribusi SalePrice (Asli) - Right Skewed", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Harga Transaksi (dalam Ribuan USD)")
axes[0].set_ylabel("Frekuensi")

# Plot 2: Transformasi Logaritmik ln(y)
sns.histplot(np.log(df['SalePrice']), kde=True, ax=axes[1], color='#2ca02c', bins=60)
axes[1].set_title("Distribusi ln(SalePrice) - Mendekati Normal", fontsize=12, fontweight='bold')
axes[1].set_xlabel("ln(SalePrice)")
axes[1].set_ylabel("Frekuensi")

plt.tight_layout()
plt.show()

## 6. Analisis Korelasi & Hubungan Antarvariabel Kunci
Menelaah hubungan linear antara spesifikasi fisik rumah, kualitas material, pemandangan, dan koordinat spasial terhadap harga jual.

In [ ]:
# Seleksi variabel representatif dari 147 kolom
fitur_kunci = [
    'SalePrice', 'SqFtTotLiving', 'SqFtLot', 'Bedrooms', 'BathFullCount', 
    'BathHalfCount', 'Stories', 'BldgGrade', 'Condition', 'YrBuilt', 
    'YrRenovated', 'X', 'Y', 'WfntLocation', 'SeattleSkyline', 'LakeWashington'
]

df_korelasi = df[fitur_kunci].dropna()
matriks_corr = df_korelasi.corr(method='pearson')

plt.figure(figsize=(14, 10))
sns.heatmap(
    matriks_corr, 
    annot=True, 
    fmt='.2f', 
    cmap='coolwarm',
    center=0, 
    vmin=-1, 
    vmax=1, 
    linewidths=0.5
)
plt.title("Matriks Korelasi Pearson Fitur Kunci terhadap Harga Properti", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Peringkat korelasi terhadap target SalePrice
target_corr = matriks_corr['SalePrice'].drop('SalePrice')
target_corr = target_corr.reindex(target_corr.abs().sort_values(ascending=False).index)

plt.figure(figsize=(10, 6))
sns.barplot(
    x=target_corr.values,
    y=target_corr.index,
    hue=target_corr.index,
    palette='coolwarm',
    legend=False
)
plt.title("Kekuatan Korelasi Fitur Prediktor terhadap SalePrice", fontsize=13, fontweight='bold')
plt.xlabel("Koefisien Korelasi Pearson")
plt.ylabel("Fitur Prediktor")
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

display(target_corr.to_frame(name='Korelasi Pearson dengan SalePrice'))

## 7. Catatan Temuan & Hipotesis Awal EDA
*Ringkasan insight empiris hasil eksplorasi data:*
1. **Fitur Fisik Dominan:** Luas area hunian (`SqFtTotLiving`, $r pprox 0.70$) dan mutu konstruksi bangunan (`BldgGrade`, $r pprox 0.67$) merupakan faktor penentu utama harga properti.
2. **Karakteristik Kemencengan (*Heavy-Tailed*):** Variabel target `SalePrice` memiliki rentang ekstrem dari \$5.000 hingga \$9,4 juta dengan median \$450.000. Penggunaan transformasi logaritmik $\ln(\text{SalePrice})$ terbukti menghasilkan distribusi simetris yang mendekati kurva normal (*bell curve*).
3. **Penyelamatan Missing Data:** Kolom `ZipCode` memiliki 3.205 nilai kosong, namun variabel spasial `X` dan `Y` terisi 100% tanpa missing value, sehingga dapat dilakukan imputasi spasial presisi tinggi via K-Nearest Neighbors (KNN).
4. **Rencana Tahap Pembersihan:** Memfilter transaksi non-pasar (< \$50.000), mengonversi proyeksi koordinat WA North ke Latitude/Longitude WGS84, serta merekayasa fitur usia bangunan (`PropertyAge`) dan rasio harga per luas lantai.